# 03 — Confidence Intervals

Quantify the uncertainty around the observed lift with a confidence interval.

---
**What this notebook covers:**
- Computing the Wald confidence interval for the difference in proportions
- Changing the confidence level (95% vs 99%)
- Interpreting the interval in plain English
- Why a CI that excludes zero is a stronger signal than the p-value alone

In [ ]:
# ── path setup — run this cell first ─────────────────────────────────────────
import sys
from pathlib import Path

ROOT = Path().resolve().parent
SRC  = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

In [ ]:
from ab_testing_framework.confidence_interval import calculate_ci
from ab_testing_framework.validation import validate_input

experiment = validate_input(10000, 450, 10000, 520)

ci_95 = calculate_ci(experiment, confidence_level=0.95)
ci_99 = calculate_ci(experiment, confidence_level=0.99)

print(f'95% CI: [{ci_95.lower_bound:.4%}, {ci_95.upper_bound:.4%}]  '
      f'(margin of error: {ci_95.margin_of_error:.4%})')
print(f'99% CI: [{ci_99.lower_bound:.4%}, {ci_99.upper_bound:.4%}]  '
      f'(margin of error: {ci_99.margin_of_error:.4%})')

In [ ]:
# Does the CI exclude zero?
for ci, label in [(ci_95, '95%'), (ci_99, '99%')]:
    excludes_zero = ci.lower_bound > 0 or ci.upper_bound < 0
    direction = 'positive' if ci.lower_bound > 0 else ('negative' if ci.upper_bound < 0 else 'spans zero')
    print(f'{label} CI: excludes zero = {excludes_zero} | direction = {direction}')

In [ ]:
# Example where CI spans zero (no meaningful lift)
exp_null = validate_input(5000, 250, 5000, 255)
ci_null  = calculate_ci(exp_null)
print(f'Null example 95% CI: [{ci_null.lower_bound:.4%}, {ci_null.upper_bound:.4%}]')
print('Spans zero:', ci_null.lower_bound < 0 < ci_null.upper_bound)

### Interpretation

- The **95% CI** for the lift is approximately **[+0.27 pp, +1.13 pp]**.
  We are 95% confident the true lift is between +0.27 and +1.13 percentage points.
- Since the **entire interval is above zero**, this corroborates the z-test decision.
  A CI that merely touches or crosses zero would weaken the conclusion even if p < 0.05.
- The **99% CI** is wider (more conservative) but still excludes zero — extra reassurance.
- When both the p-value *and* the CI exclude zero, the evidence for a real effect is strong.

Proceed to `04_Effect_Size.ipynb` to assess practical significance.